In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['SimHei']
rcParams['axes.unicode_minus'] = False
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
#将数据输出结果导出
import os

In [ ]:
df=pd.read_csv('data/clean_2019-Nov.csv')
'''-------------3:RFM 用户价值分群(从R（最近购买间隔）、F（购买频次）、M（消费金额）三个维度，把用户分成 8 类（重要价值、重要发展、重要保持、重要挽留等），统计每类用户的人数占比、消费贡献，给出对应运营策略)-------------'''

# 1. 从原始行为数据直接聚合计算RFM指标
# 筛选购买行为：RFM仅针对有购买记录的用户
purchase_data = df[df['event_type'] == 'purchase'].copy()

# 源头强制转换event_time为datetime类型
purchase_data['event_time'] = pd.to_datetime(purchase_data['event_time'])

# 设定统计截止日期：取数据中最新的购买时间作为参考基准
ref_date = purchase_data['event_time'].max()

# 按用户聚合基础指标
rfm_df = purchase_data.groupby('user_id').agg(
    last_pay_time=('event_time', 'max'),  # 最近一次购买时间
    frequency=('event_time', 'count'),     # 购买次数
    monetary=('price', 'sum')              # 消费总金额
).reset_index()

# 聚合后再强制转换一次，避免groupby后类型退化
rfm_df['last_pay_time'] = pd.to_datetime(rfm_df['last_pay_time'])

# 单独计算 R：最近购买间隔天数
rfm_df['recency'] = (ref_date - rfm_df['last_pay_time']).dt.days

# 删掉临时的时间列
rfm_df.drop('last_pay_time', axis=1, inplace=True)
# 2. 分箱打分（1-5分，分数越高越好）
# 采用等距分箱，按数值范围均匀切分；若需按用户分位数分层可改用qcut+duplicates='drop'
def rank_score(series, n_bins=5, reverse=False):
    # 计算排名百分比
    rank_pct = series.rank(pct=True, method='dense')
    if reverse:
        rank_pct = 1 - rank_pct
    # 按百分比切分
    bins = [0] + [i/n_bins for i in range(1, n_bins+1)]
    labels = list(range(1, n_bins+1))
    return pd.cut(rank_pct, bins=bins, labels=labels, include_lowest=True)

# 调用分箱
rfm_df['R_score'] = rank_score(rfm_df['recency'], n_bins=5, reverse=True)
rfm_df['F_score'] = rank_score(rfm_df['frequency'], n_bins=5, reverse=False)
rfm_df['M_score'] = rank_score(rfm_df['monetary'], n_bins=5, reverse=False)
# 转成数值
rfm_df[['R_score','F_score','M_score']] = rfm_df[['R_score','F_score','M_score']].astype(int)

# 3. 8类用户分层
conditions = [
    (rfm_df['R_score'] >= 4) & (rfm_df['F_score'] >= 4),
    (rfm_df['R_score'] >= 4) & (rfm_df['F_score'] < 4),
    (rfm_df['R_score'] < 4) & (rfm_df['F_score'] >= 4),
    (rfm_df['R_score'] < 4) & (rfm_df['F_score'] < 4)
]
choices = ['重要价值用户', '重要发展用户', '重要保持用户', '重要挽留用户']
rfm_df['user_type'] = np.select(conditions, choices, default='一般用户')

# 4. 统计各类型占比
user_type_stat = rfm_df.groupby('user_type').agg(
    用户数=('user_id','count'),
    消费总金额=('monetary','sum')
).reset_index()

# 提前计算总和，加除零保护
total_user = user_type_stat['用户数'].sum()
total_amount = user_type_stat['消费总金额'].sum()

user_type_stat['人数占比'] = user_type_stat['用户数'] / total_user if total_user != 0 else 0
user_type_stat['消费占比'] = user_type_stat['消费总金额'] / total_amount if total_amount != 0 else 0
print(" RFM用户价值分群结果")
# 按消费占比降序，突出核心价值群体
user_type_stat_show = user_type_stat.sort_values('消费占比', ascending=False).reset_index(drop=True)

# 格式化输出
print(user_type_stat_show.to_string(
    index=False,
    formatters={
        '人数占比': '{:.2%}'.format,
        '消费占比': '{:.2%}'.format,
        '消费总金额': '{:.2f}'.format
    }
))
# ====================== 保存数据用于后续可视化输出 ======================
user_type_stat_show.to_csv('output/data/rfm_segment.csv',
                           index=False, encoding='utf-8-sig')